# Warehouse Robot Navigation with Tabular Reinforcement Learning
## Q-learning (off-policy) vs SARSA (on-policy), both implemented from scratch

**Case study — Reinforcement Learning**

**Research question.** *How do Q-learning and SARSA compare when a warehouse robot must learn the
shortest collision-free route to a delivery point?*

**Dataset.** `zoya77/robot-path-planning-navigation-dataset` (Kaggle) --- 7,980 warehouse
snapshots, each a 5x5 occupancy grid with a start cell and a goal cell.

---

### What this notebook learns from, and what it does not

This is the single most important thing to understand about applying reinforcement learning to a
Kaggle dataset, and it is stated here rather than buried.

Q-learning and SARSA **do not learn from rows of data**. They learn by acting and observing the
reward that follows. A CSV cannot answer the question *"what would have happened if the robot had
turned left instead?"* --- it only records what did happen. Something must be able to answer that
counterfactual, and that something is an environment.

| | What it is | Where it comes from |
|---|---|---|
| **Layouts** | Obstacle maps, start and goal cells | The Kaggle file, unmodified |
| **Heuristic baseline** | A move-toward-goal controller | Recovered from the file's `selected_action` |
| **Environment** | A `step()` function: movement, collisions, rewards, termination | Written in this notebook |
| **Reward constants** | Step cost, collision penalty, goal reward | **Declared modelling assumptions** |

So the warehouses are real and the robot faces genuine obstacle configurations. The cost structure
is an assumption, stated openly in Section 3 and varied in a sensitivity analysis in Section 8.
That assumption is not incidental: it is what creates the effect this study measures.

### What the data audit found

Section 1 audits every column. Briefly: the occupancy grids, start/goal cells and action encoding
are sound, but `reward`, `path_length_so_far` and `converged_flag` carry no usable signal, and the
`episode_id`/`time_step` columns do not describe coherent trajectories. Those columns are excluded,
with the evidence shown rather than asserted.

### The hypothesis

Q-learning and SARSA converge to the *same* policy on a benign grid. The difference appears only
when exploration can be costly. In a warehouse it is: the shortest route runs tight along the
racking, and an exploratory move in a narrow aisle causes a collision.

The classical prediction (Sutton and Barto, *Cliff Walking*) is therefore:

- **Q-learning**, learning the value of the *greedy* policy, converges to the optimal tight route
  and accumulates **more collisions while learning**.
- **SARSA**, learning the value of the policy it actually follows --- exploration included ---
  converges to a route with **clearance**, giving **better online performance** but a slightly
  longer final path.

This requires no injected randomness. Cliff Walking is deterministic; epsilon-greedy exploration
alone produces the effect.

### Methodology note

The predecessor study in this repository compared a BiLSTM against a Transformer and reported a
single seed per model as its principal limitation. Tabular RL is cheap enough to fix that properly:
every result here is **30 independent seeds per algorithm, paired by seed**, reported with
confidence intervals and a paired non-parametric significance test.

## 0. Configuration

`SMOKE_TEST = True` shrinks the notebook to a ~1 minute end-to-end run for checking it executes;
set it to `False` for the real experiment.

In [ ]:
SMOKE_TEST = False    # True -> 3 seeds / 300 episodes, for a fast end-to-end check

# ---- experiment ---------------------------------------------------------------
SEED          = 42
N_SEEDS       = 30         # independent runs per algorithm, paired by seed
N_EPISODES    = 3_000      # training episodes per run
TILE          = 4          # compose a TILE x TILE arrangement of real 5x5 patches
N_WAREHOUSES  = 12         # further composed floors for the generality check
MAX_STEPS     = 600        # episode cap

# ---- agent hyperparameters (identical for both algorithms) ---------------------
ALPHA         = 0.10       # learning rate
GAMMA         = 0.99       # discount factor
EPS_START     = 1.00
EPS_END       = 0.05
EPS_DECAY_FRAC= 0.60       # fraction of training over which epsilon decays

# ---- reward structure: THESE ARE MODELLING ASSUMPTIONS, not data ---------------
R_STEP        = -1.0       # every move costs time
R_BUMP        = -5.0       # nudging the racking: robot stops, no damage
R_INCIDENT    = -100.0     # entering the pedestrian walkway: safety incident
R_GOAL        = +100.0     # delivery completed
SLIP_PROB     = 0.0        # wheel slip; 0 = deterministic, as Cliff Walking is

if SMOKE_TEST:
    N_SEEDS, N_EPISODES, N_WAREHOUSES = 3, 300, 3

DATA_DIR = "data/warehouse"
ART_DIR  = "artifacts/rl"

ACTIONS = ["up", "down", "left", "right"]     # recovered from the data in Section 1
N_ACTIONS = len(ACTIONS)
MOVES = {0: (-1, 0), 1: (1, 0), 2: (0, -1), 3: (0, 1)}   # (d_row, d_col)

print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL RUN'} | "
      f"{N_SEEDS} seeds x {N_EPISODES} episodes per algorithm")

In [ ]:
import json, math, os, random, time, warnings
from collections import deque, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from matplotlib.patches import Patch
from scipy import stats

warnings.filterwarnings("ignore", category=RuntimeWarning)
Path(ART_DIR).mkdir(parents=True, exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed); np.random.seed(seed)

set_seed()
print(f"numpy {np.__version__} | pandas {pd.__version__}")

In [ ]:
# ---- plotting style ------------------------------------------------------------
# Colour is assigned by algorithm and never reassigned; the same two slots are used
# in every figure so the reader learns the mapping once.
SERIES  = {"Q-learning": "#2a78d6", "SARSA": "#eb6834", "Heuristic": "#1baf7a",
           "Optimal (BFS)": "#6b7a8f"}
INK     = {"primary": "#0b0b0b", "secondary": "#52514e", "muted": "#8a8880"}
GRID_C, SURFACE = "#e3e2de", "#fcfcfb"
BLUE_SEQ = LinearSegmentedColormap.from_list("blue_seq",
    ["#fcfcfb", "#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"])

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": GRID_C, "axes.linewidth": 0.8, "axes.labelcolor": INK["secondary"],
    "axes.titlecolor": INK["primary"], "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 12, "axes.labelsize": 10,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": INK["secondary"], "ytick.color": INK["secondary"],
    "xtick.labelsize": 9, "ytick.labelsize": 9,
    "grid.color": GRID_C, "grid.linewidth": 0.8,
    "legend.frameon": False, "legend.fontsize": 9, "legend.labelcolor": INK["secondary"],
    "lines.linewidth": 2.0, "lines.markersize": 7,
    "font.size": 10, "figure.dpi": 110, "savefig.dpi": 150, "savefig.bbox": "tight",
})

def recede(ax, axis="y"):
    ax.grid(axis=axis, zorder=0); ax.set_axisbelow(True); return ax

def save_fig(fig, name):
    p = Path(ART_DIR) / name
    fig.savefig(p); print(f"saved -> {p}")

## 1. Data acquisition and audit

The dataset is pulled through the Kaggle API so acquisition is reproducible. The newer `KGAT_`
token format is read from `~/.kaggle/access_token`.

The audit below is not a formality. Several columns in this file look meaningful and are not, and
using them would silently corrupt the study. Each exclusion is justified with evidence.

In [ ]:
def ensure_dataset(data_dir=DATA_DIR):
    csv = Path(data_dir) / "grid_robot_navigation_dataset.csv"
    if csv.exists():
        print(f"Found cached dataset at {csv}"); return csv
    token = Path.home() / ".kaggle" / "access_token"
    legacy = Path.home() / ".kaggle" / "kaggle.json"
    if not (token.exists() or legacy.exists() or os.environ.get("KAGGLE_API_TOKEN")):
        raise RuntimeError(
            "No Kaggle credentials found. Either:\n"
            "  mkdir -p ~/.kaggle && echo KGAT_xxx > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token\n"
            "  export KAGGLE_API_TOKEN=KGAT_xxx\n"
            "Get a token from https://www.kaggle.com/settings")
    from kaggle.api.kaggle_api_extended import KaggleApi
    api = KaggleApi(); api.authenticate()
    Path(data_dir).mkdir(parents=True, exist_ok=True)
    api.dataset_download_files("zoya77/robot-path-planning-navigation-dataset",
                               path=data_dir, unzip=True, quiet=True)
    print(f"Downloaded to {data_dir}/")
    return csv

CSV = ensure_dataset()
raw = pd.read_csv(CSV)
GRID_COLS = [f"obs_grid_flat_{i}" for i in range(25)]
print(f"\nRaw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} columns")

In [ ]:
# ---- Audit 1: are the 'episodes' real trajectories? ----------------------------
dup = int(raw.duplicated(["episode_id", "time_step"]).sum())
by_ep = raw.groupby("episode_id")
grid_varies  = not bool((by_ep[GRID_COLS].nunique().max(axis=1) == 1).all())
start_varies = not bool((by_ep[["start_x", "start_y"]].nunique().max(axis=1) == 1).all())

print("AUDIT 1 - do episode_id / time_step describe real trajectories?")
print(f"  rows duplicating the (episode_id, time_step) key : {dup:,} of {len(raw):,}")
print(f"  obstacle grid changes within a single episode    : {grid_varies}")
print(f"  start cell changes within a single episode       : {start_varies}")
print("  => VERDICT: these are independent snapshots, not trajectories.")
print("     episode_id and time_step are excluded.\n")

# ---- Audit 2: do reward / path_length carry task signal? -----------------------
def bfs_dist(grid, start, goal):
    # Exact shortest path on a unit-cost 4-connected grid; -1 if unreachable.
    if start == goal: return 0
    H, W = grid.shape
    q, seen = deque([(start, 0)]), {start}
    while q:
        (r, c), d = q.popleft()
        for dr, dc in MOVES.values():
            nr, nc = r + dr, c + dc
            if 0 <= nr < H and 0 <= nc < W and grid[nr, nc] == 0 and (nr, nc) not in seen:
                if (nr, nc) == goal: return d + 1
                seen.add((nr, nc)); q.append(((nr, nc), d + 1))
    return -1

grids5 = raw[GRID_COLS].to_numpy().reshape(-1, 5, 5)
starts = list(zip(raw.start_y, raw.start_x))
goals  = list(zip(raw.goal_y,  raw.goal_x))
true_d = np.array([bfs_dist(grids5[i], starts[i], goals[i]) for i in range(len(raw))])
ok = true_d >= 0

print("AUDIT 2 - do the supplied reward / path columns carry task signal?")
for col in ["reward", "path_length_so_far"]:
    r = np.corrcoef(raw[col].to_numpy()[ok], true_d[ok])[0, 1]
    print(f"  corr({col:<20}, true BFS distance) = {r:+.4f}")
print(f"  converged_flag unique values: {raw.converged_flag.unique().tolist()}")
print("  => VERDICT: no correlation with the actual task. All three excluded.")
print("     Ground truth comes from BFS, computed here and verified in Section 5.\n")

# ---- Audit 3: what the grids DO give us ----------------------------------------
on_obstacle = sum(grids5[i][starts[i]] or grids5[i][goals[i]] for i in range(len(raw)))
print("AUDIT 3 - the occupancy grids, start and goal cells")
print(f"  distinct 5x5 layouts            : {raw[GRID_COLS].drop_duplicates().shape[0]:,}")
print(f"  mean obstacles per 5x5 grid     : {grids5.sum(axis=(1,2)).mean():.2f} of 25")
print(f"  start or goal placed on a shelf : {on_obstacle}  (must be 0)")
print(f"  layouts with a reachable goal   : {ok.mean():.1%}")
print("  => VERDICT: sound. These are the usable content of the dataset.")

In [ ]:
# ---- Audit 4: recover the action encoding from the data ------------------------
# selected_action is an integer 0-3 with no documented meaning. If the logged controller
# moves toward the goal, the mean offset to the goal under each action reveals the encoding.
print("AUDIT 4 - recovering the action encoding")
print(f"  action counts: {np.bincount(raw.selected_action).tolist()}")
dy, dx = raw.delta_y.to_numpy(), raw.delta_x.to_numpy()
recovered = {}
for a in range(N_ACTIONS):
    m = raw.selected_action.to_numpy() == a
    mdy, mdx = dy[m].mean(), dx[m].mean()
    name = ("up" if mdy < -0.5 else "down" if mdy > 0.5 else
            "left" if mdx < -0.5 else "right" if mdx > 0.5 else "?")
    recovered[a] = name
    print(f"  action {a}: mean offset to goal (dy={mdy:+.2f}, dx={mdx:+.2f})  ->  {name}")

assert [recovered[a] for a in range(N_ACTIONS)] == ACTIONS, \
    f"recovered encoding {recovered} does not match assumed {ACTIONS}"
print(f"  => Encoding confirmed: {ACTIONS}")
print("     The logged controller moves toward the goal; it is used as a baseline in Section 7.")

# Long-term distance statistics, used to justify the composed warehouse in Section 2.
print(f"\n  BFS distance on the raw 5x5 layouts: median {np.median(true_d[ok]):.0f}, "
      f"90th pct {np.percentile(true_d[ok], 90):.0f}, max {true_d[ok].max()}")
print(f"  layouts needing 6+ steps: {(true_d[ok] >= 6).mean():.1%}")